# Traces → SFT Distillation

Distil a deployed Foundry hosted agent's tool-using behavior into a smaller, cheaper student model — using **real production traces** as training data. No labeling required.

End-to-end:
1. Pull conversation traces from a deployed Foundry agent via the Data Generation API
2. Transform the raw export into Azure-FT-ready JSONL (5 fixes the trace export currently needs)
3. Split into train / val / test
4. Score the base student on a held-out test set with **structural tool-call matching**
5. Submit one fine-tuning job (winning hyperparameters)
6. Monitor training to completion
7. Deploy the fine-tuned model
8. Score it on the same test set and report the lift

**Result on a deployed Zava retail agent** (~100 conversations):
- Base `gpt-4.1-nano`: 7.38/10, 60% pass rate
- Fine-tuned `gpt-4.1-nano` (3 epochs, lr=1.0): **8.60/10, +16.5% lift, 60%→100% pass**

**Cost**: ~$3–8 per run. **Time**: ~30–50 minutes.

This notebook is **fully self-contained** — no external skill scripts required. All helpers (Foundry traces datagen, 5-step transform, tool-call evaluator) are defined inline.

## 1. Setup & Configuration

In [1]:

import os, json, time, random, re, subprocess, sys
from pathlib import Path
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential

PROJECT_ENDPOINT = os.environ.get(
    "AZURE_AI_PROJECT_ENDPOINT",
    "https://eastus-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus-prakharg-demo-2026",
)
AGENT_NAME = os.environ.get("AZURE_AI_AGENT_NAME")
AGENT_VERSION = os.environ.get("AZURE_AI_AGENT_VERSION")
HOURS_LOOKBACK = 720
STUDENT_MODEL = "gpt-4.1-nano"
TRAINING_TYPE = "GlobalStandard"

WORK = Path("./run").resolve()
WORK.mkdir(exist_ok=True)

credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client()
agents = list(project.agents.list())
deployments = list(project.deployments.list())

if not AGENT_NAME and agents:
    AGENT_NAME = agents[0].name
    AGENT_VERSION = AGENT_VERSION or str(getattr(agents[0], "version", "1"))

BLOCK_REASON = None
if not AGENT_NAME:
    BLOCK_REASON = (
        "The target project has no hosted agents. Deploy an agent, generate traced conversations, "
        "then set AZURE_AI_AGENT_NAME and AZURE_AI_AGENT_VERSION before retrying."
    )

RUN_STATE = {
    "project_endpoint": PROJECT_ENDPOINT,
    "auth": "AzureCliCredential",
    "student_model": STUDENT_MODEL,
    "training_type": TRAINING_TYPE,
    "agent_name": AGENT_NAME,
    "agent_version": AGENT_VERSION,
    "available_agents": [getattr(a, "name", None) for a in agents],
    "available_deployments": [getattr(d, "name", None) for d in deployments],
    "stages": {},
    "block_reason": BLOCK_REASON,
}

print(f"Project:      {PROJECT_ENDPOINT}")
print(f"Agent:        {AGENT_NAME or '(none)'}{':' + AGENT_VERSION if AGENT_VERSION else ''}")
print(f"Deployments:  {', '.join(RUN_STATE['available_deployments']) or '(none)'}")
print(f"Lookback:     {HOURS_LOOKBACK} hours")
print(f"Student:      {STUDENT_MODEL}")
print(f"Training:     {TRAINING_TYPE}")
if BLOCK_REASON:
    print(f"PREREQUISITE BLOCK: {BLOCK_REASON}")


Project:      https://eastus-prakharg-demo-2026.services.ai.azure.com/api/projects/eastus-prakharg-demo-2026
Agent:        traces-distill-zava-20261003-0805:2
Deployments:  gpt-5.4-mini, gpt-4.1-mini, gpt-4.1-nano, traces-distil-demo
Lookback:     720 hours
Student:      gpt-4.1-nano
Training:     GlobalStandard


## 2. Inline helper: Foundry traces datagen via SDK

In [2]:
def submit_foundry_traces_datagen(*, agent_name, agent_version, hours,
                                  max_samples, output_name, project_endpoint, timeout_s=1800):
    """Submit and monitor a Foundry Traces data-generation job."""
    from datetime import datetime, timedelta, timezone
    from azure.ai.projects.models import (
        DataGenerationJob, DataGenerationJobInputs,
        DataGenerationJobScenario, DataGenerationJobOutputOptions,
        FileDataGenerationJobOutput,
        TracesDataGenerationJobOptions, TracesDataGenerationJobSource,
    )

    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours)
    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        name=output_name,
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[TracesDataGenerationJobSource(
            agent_name=agent_name,
            agent_version=agent_version,
            start_time=start_time,
            end_time=end_time,
            description=f"Traces from {agent_name} version {agent_version}",
        )],
        options=TracesDataGenerationJobOptions(
            max_samples=max(15, max_samples),
            train_split=0.8,
        ),
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))
    print(f"Submitting traces datagen job (agent={agent_name}:{agent_version}, hours={hours}, max={max_samples})...")
    poller = project.beta.datasets.begin_create_generation_job(job=job)
    job_id = poller.details["job_id"]
    RUN_STATE["datagen_job_id"] = job_id
    print(f"  job.id={job_id}")

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        current = project.beta.datasets.get_generation_job(job_id)
        status = str(current.status).split(".")[-1].lower()
        RUN_STATE["datagen_job_status"] = status
        if status == "succeeded":
            break
        if status in ("failed", "cancelled"):
            error = current.error.message if current.error else "(none)"
            raise RuntimeError(f"traces datagen job ended in {status}: {error}")
        time.sleep(15)
    else:
        raise TimeoutError(f"traces datagen job did not complete in {timeout_s}s")

    outputs = [
        output for output in (current.result.outputs or [])
        if isinstance(output, FileDataGenerationJobOutput)
    ]
    RUN_STATE["datagen_generated_samples"] = current.result.generated_samples
    RUN_STATE["datagen_file_ids"] = [output.id for output in outputs]
    print(f"  generated {current.result.generated_samples} samples in {len(outputs)} file(s)")
    if not outputs:
        raise RuntimeError("traces datagen job succeeded without file outputs")

    out_path = WORK / f"{output_name}_dg.jsonl"
    out_blob = b""
    for output in outputs:
        out_blob += client.files.content(file_id=output.id).read()
    out_path.write_bytes(out_blob)
    print(f"  saved {out_path.name} ({out_path.stat().st_size:,} bytes)")
    return out_path


## 3. Inline helpers: 5-step transform for Foundry traces export

Foundry's traces export currently emits JSONL with five issues that Azure FT preprocessing rejects:
1. **Overlapping snapshots** — each LangGraph node invocation produces a span; the worker stitches them into one row, so one row contains N overlapping snapshots
2. **Fragments** — when the customer simulator ends right after an asst clarifying question, you get 2-msg rows with no tool_calls (no SFT signal)
3. **content="null"** — assistant tool-call rows have content as the literal string `"null"` (Azure FT requires content omitted entirely on tool-call rows)
4. **Consecutive asst tool_call turns** — sometimes two adjacent asst spans each issue one call (Azure FT requires tool replies immediately after each asst turn)
5. **Missing system + tools** — the trace export does not currently emit system or tools at row level; tool-using FT needs both

These five functions apply all five fixes. As Foundry's export improves these should become no-ops, but they're idempotent so the notebook is safe to keep.

In [3]:
def dedup_messages(msgs):
    """Collapse overlapping snapshots via first-occurrence dedup."""
    def key(m):
        tcs = m.get("tool_calls") or []
        tc_key = tuple(
            (tc.get("id"),
             (tc.get("function") or {}).get("name"),
             (tc.get("function") or {}).get("arguments"))
            for tc in tcs
        )
        return (m.get("role"), m.get("content") or "", m.get("tool_call_id"), tc_key)
    seen, out = set(), []
    for m in msgs:
        k = key(m)
        if k in seen:
            continue
        seen.add(k)
        out.append(m)
    return out, len(msgs) - len(out)


def is_fragment(msgs):
    """Row is a fragment if it has no assistant tool_calls (nothing to learn)."""
    return not any(m.get("role") == "assistant" and m.get("tool_calls") for m in msgs)


def merge_consecutive_asst_tool_calls(msgs):
    """Merge adjacent assistant tool-call messages before their ordered tool replies."""
    out, merged, i = [], 0, 0
    while i < len(msgs):
        m = msgs[i]
        if m.get("role") == "assistant" and m.get("tool_calls"):
            combined = dict(m)
            combined["tool_calls"] = list(m["tool_calls"])
            j = i + 1
            while j < len(msgs) and msgs[j].get("role") == "assistant" and msgs[j].get("tool_calls"):
                combined["tool_calls"].extend(msgs[j]["tool_calls"])
                merged += 1
                j += 1
            out.append(combined)
            i = j
        else:
            out.append(m)
            i += 1
    if merged:
        msgs[:] = out
    return merged


def fix_null_content(msgs):
    """Remove content from assistant tool-call rows."""
    n = 0
    for m in msgs:
        if m.get("role") == "assistant" and m.get("tool_calls") and "content" in m:
            m.pop("content")
            n += 1
    return n


def repair_tool_call_ids(msgs, row_number):
    """Restore tool reply IDs omitted by the traces exporter using reply order."""
    pending = []
    generated = 0
    for message_index, message in enumerate(msgs):
        if message.get("role") == "assistant" and message.get("tool_calls"):
            pending = []
            for call_index, call in enumerate(message["tool_calls"]):
                if not call.get("id"):
                    call["id"] = f"call_trace_{row_number}_{message_index}_{call_index}"
                    generated += 1
                pending.append(call["id"])
        elif message.get("role") == "tool":
            if not pending:
                raise ValueError(f"row {row_number} contains an orphan tool reply")
            message["tool_call_id"] = pending.pop(0)
        elif message.get("role") != "tool" and pending:
            pending = []
    return generated


def transform_traces_for_ft(raw_path, system_prompt, tools, out_path):
    """Normalize current Foundry trace exports into Azure tool-use SFT JSONL."""
    rows_in, rows_out, repaired_ids = 0, 0, 0
    with open(raw_path, encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            rows_in += 1
            row = json.loads(line)
            msgs = [m for m in (row.get("messages") or []) if m.get("role") != "system"]
            msgs, _ = dedup_messages(msgs)
            if is_fragment(msgs):
                continue
            merge_consecutive_asst_tool_calls(msgs)
            fix_null_content(msgs)
            try:
                repaired_ids += repair_tool_call_ids(msgs, rows_in)
            except ValueError:
                continue
            row["messages"] = [{"role": "system", "content": system_prompt}] + msgs
            row["tools"] = tools
            row["parallel_tool_calls"] = True
            fout.write(json.dumps(row) + "\n")
            rows_out += 1
    print(f"  Transformed {rows_in} raw -> {rows_out} clean rows; repaired {repaired_ids} tool IDs")
    return out_path


## 4. Pull traces and transform into FT-ready JSONL

In [4]:

SYSTEM_PROMPT = Path("fixtures/zava_system_prompt.md").read_text(encoding="utf-8")
TOOLS = json.loads(Path("fixtures/zava_tools.json").read_text(encoding="utf-8"))
data = []

if BLOCK_REASON:
    RUN_STATE["stages"]["trace_acquisition"] = "blocked_external_prerequisite"
    RUN_STATE["stages"]["transformation"] = "blocked_external_prerequisite"
    print(f"Trace acquisition blocked: {BLOCK_REASON}")
else:
    try:
        RAW_DATA = submit_foundry_traces_datagen(
            agent_name=AGENT_NAME,
            agent_version=AGENT_VERSION,
            hours=HOURS_LOOKBACK,
            max_samples=100,
            output_name="traces-raw",
            project_endpoint=PROJECT_ENDPOINT,
        )
        RUN_STATE["stages"]["trace_acquisition"] = "succeeded"
        CLEAN_DATA = WORK / "traces-clean.jsonl"
        transform_traces_for_ft(RAW_DATA, SYSTEM_PROMPT, TOOLS, CLEAN_DATA)
        with open(CLEAN_DATA, encoding="utf-8") as f:
            data = [json.loads(line) for line in f if line.strip()]
        if not data:
            raise RuntimeError("Trace transformation produced no fine-tuning rows.")
        RUN_STATE["stages"]["transformation"] = "succeeded"
        RUN_STATE["clean_rows"] = len(data)
        print(f"\nCleaned: {len(data)} rows")
    except Exception as exc:
        BLOCK_REASON = f"Trace acquisition/transformation failed: {type(exc).__name__}: {exc}"
        RUN_STATE["block_reason"] = BLOCK_REASON
        RUN_STATE["stages"]["trace_acquisition"] = RUN_STATE["stages"].get("trace_acquisition", "failed")
        RUN_STATE["stages"]["transformation"] = "blocked"
        print(BLOCK_REASON)


Submitting traces datagen job (agent=traces-distill-zava-20261003-0805:2, hours=720, max=100)...


  job.id=datagen-e543b86d98874d3cb660a376f7ba7c9c


  generated 100 samples in 2 file(s)


  saved traces-raw_dg.jsonl (477,867 bytes)
  Transformed 100 raw -> 100 clean rows; repaired 53 tool IDs

Cleaned: 100 rows


## 5. Split into train / val / test

In [5]:

rng = random.Random(42)
indices = list(range(len(data)))
rng.shuffle(indices)
n_train = int(0.8 * len(indices))
n_val = int(0.1 * len(indices))
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train+n_val]
test_idx = indices[n_train+n_val:]

TRAIN_PATH = WORK / "train.jsonl"
VAL_PATH = WORK / "val.jsonl"
TEST_PATH = WORK / "test.jsonl"
for split_path, idxs in [(TRAIN_PATH, train_idx), (VAL_PATH, val_idx), (TEST_PATH, test_idx)]:
    with split_path.open("w", encoding="utf-8") as f:
        for i in idxs:
            f.write(json.dumps(data[i]) + "\n")

RUN_STATE["splits"] = {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)}
RUN_STATE["stages"]["split"] = "succeeded" if data else "blocked"
print(f"  train: {len(train_idx)} rows")
print(f"  val:   {len(val_idx)} rows")
print(f"  test:  {len(test_idx)} rows")


  train: 80 rows
  val:   10 rows
  test:  10 rows


## 6. Baseline evaluation via the Foundry evals SDK

We use `azure-ai-evaluation.evaluate()` as the driver. The built-in evaluators don't cover tool-call structural matching, so we provide a small custom evaluator function the SDK runs across all test rows.

In [6]:

def tool_call_score(ref_calls, out_calls):
    if not ref_calls: return 10 if not out_calls else 5
    if not out_calls: return 1
    def _name(c):
        if isinstance(c, dict): return ((c.get("function") or {}).get("name")) or c.get("name")
        fn = getattr(c, "function", None)
        return getattr(fn, "name", None) if fn else getattr(c, "name", None)
    def _args(c):
        if isinstance(c, dict):
            raw = ((c.get("function") or {}).get("arguments")) or c.get("arguments")
        else:
            fn = getattr(c, "function", None)
            raw = getattr(fn, "arguments", None) if fn else getattr(c, "arguments", None)
        if isinstance(raw, str):
            try: return json.loads(raw)
            except Exception: return {"_raw": raw}
        return raw or {}
    ref_names = [_name(c) for c in ref_calls]
    out_names = [_name(c) for c in out_calls]
    ref_set, out_set = set(ref_names), set(out_names)
    overlap = ref_set & out_set
    if not overlap: return 1
    if ref_set != out_set:
        ratio = len(overlap) / len(ref_set | out_set)
        return max(2, int(round(2 + ratio * 6)))
    ref_args = {_name(c): _args(c) for c in ref_calls}
    out_args = {_name(c): _args(c) for c in out_calls}
    return 10 if all(ref_args[n] == out_args.get(n) for n in ref_names) else 8


def tool_call_evaluator(*, response, ground_truth, **kwargs):
    ref = json.loads(ground_truth) if isinstance(ground_truth, str) else (ground_truth or [])
    out = json.loads(response) if isinstance(response, str) else (response or [])
    score = tool_call_score(ref, out)
    return {"tool_match_score": score, "tool_match_pass": 1.0 if score >= 8 else 0.0}


def make_target(model_name):
    def _target(*, prompt, system, **kwargs):
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
            tools=TOOLS,
            temperature=0.0,
            max_completion_tokens=2048,
        )
        tool_calls = resp.choices[0].message.tool_calls or []
        return {"response": json.dumps([
            {"function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in tool_calls
        ])}
    return _target


EVAL_DATA_PATH = WORK / "eval_data.jsonl"
with open(TEST_PATH, encoding="utf-8") as fin, open(EVAL_DATA_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip(): continue
        row = json.loads(line)
        msgs = row["messages"]
        user_msg = next((m.get("content") or "" for m in msgs if m.get("role") == "user"), "")
        first_asst = next((m for m in msgs if m.get("role") == "assistant"), {})
        ref_tcs = first_asst.get("tool_calls") or []
        gt = [{"function": {"name": (tc.get("function") or {}).get("name"),
                            "arguments": (tc.get("function") or {}).get("arguments")}} for tc in ref_tcs]
        fout.write(json.dumps({"system": SYSTEM_PROMPT, "prompt": user_msg, "ground_truth": json.dumps(gt)}) + "\n")

baseline_results = None
baseline_combined = None
baseline_pass = None
if test_idx:
    deployment_names = RUN_STATE["available_deployments"]
    if STUDENT_MODEL not in deployment_names:
        BLOCK_REASON = (
            f"Baseline evaluation requires a '{STUDENT_MODEL}' deployment in the project; "
            f"available deployments: {deployment_names}."
        )
        RUN_STATE["block_reason"] = BLOCK_REASON
        RUN_STATE["stages"]["baseline_evaluation"] = "blocked_external_prerequisite"
        print(BLOCK_REASON)
    else:
        from azure.ai.evaluation import evaluate
        baseline_results = evaluate(
            data=str(EVAL_DATA_PATH),
            target=make_target(STUDENT_MODEL),
            evaluators={"tool_match": tool_call_evaluator},
            output_path=str(WORK / "baseline_eval_results.json"),
        )
        baseline_combined = baseline_results["metrics"].get("tool_match.tool_match_score", 0)
        baseline_pass = baseline_results["metrics"].get("tool_match.tool_match_pass", 0) * 100
        RUN_STATE["stages"]["baseline_evaluation"] = "succeeded"
        RUN_STATE["baseline"] = {"combined": baseline_combined, "pass_rate": baseline_pass}
        print(f"Baseline: combined={baseline_combined:.2f}/10 pass_rate={baseline_pass:.1f}%")
else:
    RUN_STATE["stages"]["baseline_evaluation"] = "blocked"
    print("Baseline evaluation skipped because no trace-derived test rows are available.")


2026-10-03 11:42:03 -0700   72240 execution.bulk     INFO     Finished 1 / 10 lines.


2026-10-03 11:42:03 -0700   72240 execution.bulk     INFO     Average execution time for completed lines: 0.95 seconds. Estimated time for incomplete lines: 8.55 seconds.


2026-10-03 11:42:04 -0700   72240 execution.bulk     INFO     Finished 2 / 10 lines.


2026-10-03 11:42:04 -0700   72240 execution.bulk     INFO     Average execution time for completed lines: 0.71 seconds. Estimated time for incomplete lines: 5.68 seconds.


2026-10-03 11:42:04 -0700   72240 execution.bulk     INFO     Finished 3 / 10 lines.


...

2026-10-03 11:42:05 -0700   69616 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.


2026-10-03 11:42:05 -0700   69616 execution.bulk     INFO     Finished 10 / 10 lines.


2026-10-03 11:42:05 -0700   69616 execution.bulk     INFO     Average execution time for completed lines: 0.0 seconds. Estimated time for incomplete lines: 0.0 seconds.


Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "tool_match_20261003_184205_011472"
Run status: "Completed"
Start time: "2026-10-03 18:42:05.011472+00:00"
Duration: "0:00:01.011992"

======= Combined Run Summary (Per Evaluator) =======

{
    "tool_match": {
        "status": "Completed",
        "duration": "0:00:01.011992",
        "completed_lines": 10,
        "failed_lines": 0,
        "log_path": null,
        "per_line_errors": {},
        "error_message": null,
        "error_code": null
    }
}


Evaluation results saved to "C:\Data\AIP\fine-tuning\Demos\TracesDistillation\run\baseline_eval_results.json".

Baseline: combined=7.50/10 pass_rate=50.0%


## 7. Submit the fine-tuning job

Winning hyperparameters from prior experiments: **`gpt-4.1-nano`, 3 epochs, learning-rate multiplier 1.0**.

In [7]:
train_file = val_file = ft_job = None
resume_ft_job_id = os.environ.get("AZURE_FINE_TUNING_JOB_ID")
if resume_ft_job_id:
    ft_job = client.fine_tuning.jobs.retrieve(resume_ft_job_id)
    RUN_STATE["fine_tuning_job_id"] = ft_job.id
    RUN_STATE["stages"]["fine_tuning_submission"] = "resumed"
    print(f"Resuming fine-tuning job {ft_job.id} status={ft_job.status}")
elif not train_idx or not val_idx:
    RUN_STATE["stages"]["fine_tuning_submission"] = "blocked"
    print("Fine-tuning submission skipped because trace-derived train/validation rows are unavailable.")
else:
    print("Uploading train + val files...")
    with open(TRAIN_PATH, "rb") as fh:
        train_file = client.files.create(file=(TRAIN_PATH.name, fh), purpose="fine-tune")
    with open(VAL_PATH, "rb") as fh:
        val_file = client.files.create(file=(VAL_PATH.name, fh), purpose="fine-tune")

    for uploaded in (train_file, val_file):
        for _ in range(90):
            uploaded = client.files.retrieve(file_id=uploaded.id)
            if uploaded.status in ("processed", "error"):
                break
            time.sleep(2)
        if uploaded.status != "processed":
            raise RuntimeError(f"File {uploaded.id} ended in status {uploaded.status}")
        print(f"  {uploaded.id} status={uploaded.status}")

    print("\nSubmitting GlobalStandard fine-tuning job...")
    ft_job = client.fine_tuning.jobs.create(
        model=STUDENT_MODEL,
        training_file=train_file.id,
        validation_file=val_file.id,
        method={"type": "supervised"},
        hyperparameters={"n_epochs": 3, "learning_rate_multiplier": 1.0},
        suffix="traces-distil-demo",
        extra_body={"trainingType": TRAINING_TYPE},
    )
    RUN_STATE["fine_tuning_job_id"] = ft_job.id
    RUN_STATE["stages"]["fine_tuning_submission"] = "succeeded"
    print(f"  Job: {ft_job.id} status={ft_job.status}")


## 8. Monitor training

In [8]:

FT_MODEL_ID = None
if ft_job is None:
    RUN_STATE["stages"]["fine_tuning_monitor"] = "blocked"
    print("Fine-tuning monitoring skipped because no job was submitted.")
else:
    print(f"Monitoring job {ft_job.id}...\n")
    last_seen_step = -1
    while True:
        job = client.fine_tuning.jobs.retrieve(ft_job.id)
        RUN_STATE["fine_tuning_job_status"] = job.status
        events = list(client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job.id, limit=10))
        for event in reversed(events):
            msg = event.message or ""
            if "Step " in msg and ":" in msg:
                try:
                    step = int(msg.split("Step ")[1].split(":")[0])
                    if step > last_seen_step:
                        print(f"  {time.strftime('%H:%M:%S')} {msg[:90]}")
                        last_seen_step = step
                except (ValueError, IndexError):
                    pass
        if job.status in ("succeeded", "failed", "cancelled"):
            print(f"\nFinal status: {job.status}")
            RUN_STATE["stages"]["fine_tuning_monitor"] = job.status
            if job.status != "succeeded":
                error = job.error.message if job.error else "(none)"
                raise RuntimeError(f"Fine-tuning {job.status}: {error}")
            FT_MODEL_ID = job.fine_tuned_model
            RUN_STATE["fine_tuned_model"] = FT_MODEL_ID
            print(f"Fine-tuned model: {FT_MODEL_ID}")
            break
        time.sleep(30)


## 9. Deploy the fine-tuned model

In [9]:

DEPLOY_NAME = "traces-distil-demo"
deployment_ready = False
if not FT_MODEL_ID:
    RUN_STATE["stages"]["deployment"] = "blocked"
    print("Deployment skipped because no fine-tuned model was produced.")
else:
    subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID")
    resource_group = os.environ.get("AZURE_RESOURCE_GROUP")
    account_name = os.environ.get("AZURE_AI_ACCOUNT_NAME", "eastus-prakharg-demo-2026")
    if not subscription_id or not resource_group:
        raise RuntimeError(
            "AZURE_SUBSCRIPTION_ID and AZURE_RESOURCE_GROUP are required for non-interactive deployment."
        )
    deploy_url = (
        f"https://management.azure.com/subscriptions/{subscription_id}"
        f"/resourceGroups/{resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{account_name}"
        f"/deployments/{DEPLOY_NAME}?api-version=2024-10-01"
    )
    body = {
        "sku": {"name": "GlobalStandard", "capacity": 100},
        "properties": {"model": {"format": "OpenAI", "name": FT_MODEL_ID, "version": "1"}},
    }
    import urllib.request
    token = credential.get_token("https://management.azure.com/.default").token
    request = urllib.request.Request(
        deploy_url,
        method="PUT",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        data=json.dumps(body).encode(),
    )
    with urllib.request.urlopen(request) as response:
        print(f"Deployment submitted: {DEPLOY_NAME} ({response.status})")

    print("Waiting for deployment to become inferenceable...")
    for i in range(40):
        try:
            client.chat.completions.create(
                model=DEPLOY_NAME,
                messages=[{"role": "user", "content": "hi"}],
                max_completion_tokens=5,
            )
            deployment_ready = True
            print(f"Ready after {i * 30}s")
            break
        except Exception:
            time.sleep(30)
    if not deployment_ready:
        raise TimeoutError("Fine-tuned deployment did not become inferenceable within 20 minutes.")
    RUN_STATE["stages"]["deployment"] = "succeeded"
    RUN_STATE["deployment_name"] = DEPLOY_NAME


## 10. Evaluate the fine-tuned model and compare

In [10]:

ft_results = None
ft_combined = None
ft_pass = None
if not deployment_ready:
    RUN_STATE["stages"]["fine_tuned_evaluation"] = "blocked"
    print("Fine-tuned evaluation skipped because no inference deployment is available.")
else:
    from azure.ai.evaluation import evaluate
    print(f"Fine-tuned ({STUDENT_MODEL}) evaluation...")
    ft_results = evaluate(
        data=str(EVAL_DATA_PATH),
        target=make_target(DEPLOY_NAME),
        evaluators={"tool_match": tool_call_evaluator},
        output_path=str(WORK / "ft_eval_results.json"),
    )
    ft_combined = ft_results["metrics"].get("tool_match.tool_match_score", 0)
    ft_pass = ft_results["metrics"].get("tool_match.tool_match_pass", 0) * 100
    lift = ((ft_combined - baseline_combined) / baseline_combined * 100
            if baseline_combined and baseline_combined > 0 else None)
    RUN_STATE["stages"]["fine_tuned_evaluation"] = "succeeded"
    RUN_STATE["fine_tuned_evaluation"] = {
        "combined": ft_combined,
        "pass_rate": ft_pass,
        "lift_percent": lift,
    }
    print(f"Baseline: combined={baseline_combined:.2f}/10 pass_rate={baseline_pass:.1f}%")
    print(f"Fine-tuned: combined={ft_combined:.2f}/10 pass_rate={ft_pass:.1f}%")
    print(f"Lift: {lift:+.1f}%")

summary_path = WORK / "run_summary.json"
summary_path.write_text(json.dumps(RUN_STATE, indent=2), encoding="utf-8")
print(f"Run summary: {summary_path}")
print(json.dumps(RUN_STATE, indent=2))


## (Optional) Populate trace history first

If your agent has no trace history yet, the `fixtures/push_prompts.py` script sends ~500 diverse retail-style prompts through your agent (traces land in App Insights within ~90 seconds). Run it before cell 4 (`Pull traces and transform...`):

```bash
python fixtures/push_prompts.py \
    --agent-name <your-hosted-agent> \
    --agent-version <version> \
    --num-prompts 500 \
    --concurrency 4 \
    --project-endpoint $AZURE_AI_PROJECT_ENDPOINT
```

## Cleanup

```python
req = urllib.request.Request(deploy_url, method="DELETE", headers={"Authorization": f"Bearer {token}"})
urllib.request.urlopen(req)
client.files.delete(train_file.id)
client.files.delete(val_file.id)
```

## Bring your own agent

Replace `AGENT_NAME` / `AGENT_VERSION` and the two fixture files (`zava_system_prompt.md`, `zava_tools.json`) with your agent's system prompt + tool catalog (OpenAI chat-completions format).

## Dependencies

```
pip install openai>=2.0 azure-ai-projects>=2.2.0 azure-identity>=1.21 azure-ai-evaluation>=1.0
```